# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains proteomics data, including protein abundances, modifications, and sequence attributes from human mast cell samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A Croissant dataset's main tables (record sets) and their fields (columns) are uniquely identified by their `@id`.

Let's enumerate all record sets and fields by their `@id`.

In [ ]:
# List all record set @ids and fields @ids
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets in the schema.\n")

for record_set in record_sets:
    print(f"RecordSet: {record_set.id}  [name: {record_set.name if hasattr(record_set, 'name') else ''}]")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"  Field: {field.id} [name: {field.name if hasattr(field, 'name') else ''}, dataType: {getattr(field, 'data_type', None)}]")
    print()

## 2a. Preview Records
Fetch and print a few records from one available record set for a quick glance at the data format using their `@id`.

_Update `record_set_id` below to the actual record set you want to preview from the above output._

In [ ]:
# Example: Preview the first few records from the main record set
# Replace the record set ID below with the actual @id printed above, e.g. 'cr:RecordSet/Proteins'
example_record_set_id = record_sets[0].id if len(record_sets) else None
if example_record_set_id:
    print(f"Preview rows for record set: {example_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i > 3:
            break
else:
    print('No record sets found. Check schema or data loading step.')

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis.

Below, we use the `@id` of record sets. Use the overview above to choose appropriate `@id`s (e.g., `'cr:RecordSet/Proteins'`).

In [ ]:
# List of record set IDs to extract (update with relevant @ids for your dataset)
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records from: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"    #columns: {list(df.columns)}")
        print(df.head(2))
    else:
        print("    (No records loaded)")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalizing, and grouping.

**Note:** Please update `<record_set_id>`, `<numeric_field>`, and `<group_field>` as needed for your use case using the summary above.

In [ ]:
# Example usage with placeholder field names -- update these as appropriate:
import numpy as np

# Choose a record set to analyze
record_set_id = example_record_set_id
df = dataframes.get(record_set_id)
if df is not None:
    # Choose a numeric field (must be present in df.columns)
    # Example: 'abundance' or 'intensity' or similar - update below
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print('No numeric fields detected; update with an available field name.')
    else:
        numeric_field = numeric_fields[0]
        print(f'Using numeric field: {numeric_field}')

        threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f} (n={len(filtered_df)}):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Choose a grouping field (e.g. 'sample_id' or 'description') if present
        group_field = None
        group_candidate_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() < len(df)//2]
        if group_candidate_fields:
            group_field = group_candidate_fields[0]
            print(f'\nGrouping by field: {group_field}')
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print('No suitable grouping field found.')
else:
    print('DataFrame is empty or missing.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot a histogram and a boxplot of a selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    field = numeric_fields[0]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {field}")
    plt.xlabel(field)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[field])
    plt.title(f"Boxplot of {field}")

    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[field])
        plt.title(f"{field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(field)
        plt.show()
else:
    print('No numeric data for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and profile a [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using Python and the `mlcroissant` library.

- We inspected the metadata, available record sets, and fields (all referenced by their `@id`).
- We extracted records into pandas DataFrames and explored their shape and basic summary statistics.
- Standard EDA operations such as filtering and normalization were applied to selected fields.
- We produced example visualizations for numeric fields.

Further steps might include integrating with downstream analysis pipelines or machine learning workflows.

**References:**
- [mlcroissant documentation](https://mlcommons.github.io/croissant/python/)
- [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)